# 02 · MCP for AI agents

## What is MCP?

**MCP (Model Context Protocol)** is one standard interface for agents to talk to any external tool or data source.

- **Without MCP:** every tool needs custom glue code. A different integration for Google, another for the filesystem, another for Wikipedia. Each one breaks differently.
- **With MCP:** every tool speaks the same language. Any MCP-compatible server just works.

## Three components

- **Host** is the application. It owns the agent and starts everything.
  - Host = the app that runs the agent and manages everything. It creates the clients.
- **Client** lives inside the host, one per server. Talks to its server via JSON-RPC 2.0 messages.
  - Client = "please do this" (asks). Lives inside the host.
- **Server** exposes one capability. Receives the message, runs the function, returns the result.
  - Server = "done, here's the result" (does). Lives in the tool's world.

MCP is what lets them talk across that boundary.

## JSON-RPC 2.0 message workflow

Each request is a JSON object with a method, params, and an id. The response matches the same id back.

```mermaid
sequenceDiagram
    participant A as Agent (Host)
    participant C as Client
    participant S as Server

    A->>C: call search tool
    C->>S: {"method": "search", "params": {"query": "..."}, "id": 1}
    S-->>C: {"result": ["article 1", "article 2"], "id": 1}
    C-->>A: returns results
```

## Example: research agent

Goal: *"Research transformer models and save a summary to my desktop."*

```mermaid
flowchart TD
    U([User]) -->|"Research transformer models\nand save a summary"| H

    subgraph H["Host (Python script)"]
        A[Agent / LLM]
        C1[Web Search Client]
        C2[Wikipedia Client]
        C3[Filesystem Client]
    end

    C1 <-->|JSON-RPC| S1[Web Search Server]
    C2 <-->|JSON-RPC| S2[Wikipedia Server]
    C3 <-->|JSON-RPC| S3[Filesystem Server]

    S1 --> F1[Search API]
    S2 --> F2[Wikipedia API]
    S3 --> F3[summary.txt on Desktop]

    A --> C1
    A --> C2
    A --> C3
```

## Setup

In [ ]:
import os

from agents import Agent, ModelSettings, Runner, trace
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
print("API key found:", bool(os.environ.get("OPENAI_API_KEY")))